# e-SNLI Batch Denoising — Corpus-level Continuous TF-IDF

In [12]:
import sys, os

sys.path.insert(0, os.path.dirname(os.getcwd()))

import random

import torch
from src.configs import SAEConfig
from src.denoiser import Denoiser
from src.configs import DenoisingConfig
from src.utils.activations_utils import top_k_features_per_token
from src.utils.visualization import plot_per_token_topk_heatmap

## Load activations from disk

In [2]:
DATA_PATH = "../data/activations/esnli-l40-65k.pt"

data = torch.load(DATA_PATH, weights_only=False)

sae_encodings = data["sae_encodings"]
recon_stats = data["recon_stats"]
sequences = data["sequence"]
prompt_char_lens = data["prompt_lens"]
generation_token_ids = data.get("generation_token_ids")
dataset_info = data["dataset_info"]
sae_cfg_dict = data["sae_config"]

sae_config = SAEConfig(**sae_cfg_dict)

n_prompts = len(sae_encodings)
total_tokens = sum(enc.shape[0] for enc in sae_encodings)
n_features = sae_encodings[0].shape[1]
mean_fvu = sum(s["fvu"] for s in recon_stats) / len(recon_stats)
mean_l0 = sum(s["l0"] for s in recon_stats) / len(recon_stats)

print(f"Prompts:         {n_prompts}")
print(f"Total tokens:    {total_tokens}")
print(f"Features (d_sae): {n_features}")
print(f"Mean FVU:        {mean_fvu:.4f}")
print(f"Mean L0:         {mean_l0:.1f}")
print(f"SAE config:      {sae_config}")

Prompts:         500
Total tokens:    72852
Features (d_sae): 65536
Mean FVU:        0.0229
Mean L0:         57.6
SAE config:      SAEConfig(repo_id='google/gemma-scope-2-27b-it', sae_type='resid_post', layer=40, width='65k', l0='medium')


In [3]:
print([k for k in data.keys()])

['sae_encodings', 'recon_stats', 'sequence', 'prompt_lens', 'generation_token_ids', 'dataset_info', 'sae_config']


In [4]:
print(data["sae_encodings"][0])


tensor(indices=tensor([[    0,     0,     0,  ...,   148,   148,   148],
                       [  142,   240,   411,  ..., 38911, 52942, 62451]]),
       values=tensor([574.1371, 442.1205, 491.8810,  ..., 612.6995,
                      512.8052, 574.3268]),
       size=(149, 65536), nnz=9691, layout=torch.sparse_coo)


## Compute corpus-level IDF, sample one prompt, and denoise

In [13]:
THRESHOLD = 10.0

idf, corpus_total = Denoiser.compute_corpus_idf(sae_encodings, threshold=THRESHOLD)
print(f"Corpus total tokens: {corpus_total}")
print(f"IDF vector shape:    {idf.shape}")
print(f"IDF range:           [{idf.min():.3f}, {idf.max():.3f}]")

seed = random.randint(0, 2**32 - 1)
random.seed(seed)
sample_idx = random.randint(0, n_prompts - 1)
print(f"\nRandom seed: {seed}")
print(f"Sampled prompt index: {sample_idx}")

sample_sparse = sae_encodings[sample_idx]
sample_acts = sample_sparse.to_dense().float()
denoised_acts = Denoiser.apply_tfidf(sample_acts, idf)

denoising_config = DenoisingConfig(method="continuous_tfidf", params={"threshold": 10.0})
denoiser = Denoiser()
locally_denoised_acts = denoiser.denoise(sample_acts, denoising_config)

gen_text = sequences[sample_idx][prompt_char_lens[sample_idx]:]
ground_truth = dataset_info["ground_truths"][sample_idx]

print(f"\nGround truth label: {ground_truth}")
print(f"Generation tokens:  {sample_acts.shape[0]}")
print(f"Generation text:    {gen_text[:200]}...")

Computing corpus IDF: 100%|██████████| 500/500 [00:39<00:00, 12.80it/s]


Corpus total tokens: 72852
IDF vector shape:    torch.Size([65536])
IDF range:           [0.955, 11.196]

Random seed: 3501968139
Sampled prompt index: 317

Ground truth label: neutral
Generation tokens:  145
Generation text:    
<reasoning>
The premise states a specific action - a boy touching the propeller. The hypothesis states a more general situation - a boy admiring the aircraft with a friend. Touching the propeller cou...


## HuggingFace Authentication

In [14]:
import sys
from huggingface_hub import login

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import google.colab.userdata
    hf_token = google.colab.userdata.get("HFWrite")
    login(token=hf_token)
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")
    login(token=hf_token)
        

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Tokenize the sampled generation for heatmap labels

In [15]:
MODEL_NAME = "google/gemma-3-27b-it"

try:
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=True)

    if generation_token_ids is not None:
        gen_ids = generation_token_ids[sample_idx]
        tokens = tokenizer.convert_ids_to_tokens(gen_ids)
    else:
        token_ids = tokenizer.encode(gen_text, add_special_tokens=False)
        tokens = tokenizer.convert_ids_to_tokens(token_ids)

    if len(tokens) != denoised_acts.shape[0]:
        print(
            f"Token count mismatch: tokenizer produced {len(tokens)}, "
            f"activations have {denoised_acts.shape[0]}. "
            f"Falling back to positional labels."
        )
        tokens = [f"t_{i}" for i in range(denoised_acts.shape[0])]
except Exception as e:
    print(f"Could not load tokenizer ({e}); using positional labels.")
    tokens = [f"t_{i}" for i in range(denoised_acts.shape[0])]

print(f"Token count: {len(tokens)}")
print(f"First 10 tokens: {tokens[:10]}")

Token count: 145
First 10 tokens: ['\n', '<', 'reason', 'ing', '>', '\n', 'The', '▁premise', '▁states', '▁a']


## Top-K denoised feature heatmap

In [16]:
TOP_K = 15

top_values, top_indices = top_k_features_per_token(denoised_acts, k=TOP_K)

fig = plot_per_token_topk_heatmap(
    top_values,
    top_indices,
    tokens=tokens,
    title=f"e-SNLI Sample #{sample_idx} — Top-{TOP_K} Denoised SAE Features (corpus TF-IDF)",
)
fig.show()

## Top-K original feature heatmap

In [17]:
top_values, top_indices = top_k_features_per_token(sample_acts, k=TOP_K)

fig = plot_per_token_topk_heatmap(
    top_values,
    top_indices,
    tokens=tokens,
    title=f"e-SNLI Sample #{sample_idx} — Top-{TOP_K} Denoised SAE Features (corpus TF-IDF)",
)
fig.show()

## Top-K locally-denoised feature heatmap

In [18]:
top_values, top_indices = top_k_features_per_token(locally_denoised_acts, k=TOP_K)

fig = plot_per_token_topk_heatmap(
    top_values,
    top_indices,
    tokens=tokens,
    title=f"e-SNLI Sample #{sample_idx} — Top-{TOP_K} Denoised SAE Features (corpus TF-IDF)",
)
fig.show()

# Visualize Select Features

In [ ]:
from src.utils.visualization import summarize_latents

feature_map = summarize_latents(
    denoised_acts, tokens,
    sae_config=sae_config,
    model_name=MODEL_NAME,
    top_k=10,
    print_first_n=3,
)

In [ ]:
FEATURES_TO_INSPECT = [0, 0, 0]  # <-- fill in after reviewing heatmap

for feat_idx in FEATURES_TO_INSPECT:
    feat = feature_map[feat_idx]
    print(f"Inspecting feature {feat_idx}")
    print(f"repr: {feat!r}")
    print(f"frac_nonzero: {feat.frac_nonzero}")
    print(f"top_tokens: {feat.top_tokens()}")
    print()
    feat.inspect()

## Steering Experiment

### Load Model + SAE

In [ ]:
from src.configs import ModelConfig
from src.gemma_model import GemmaModel
from src.SAE import JumpReLUSAE

model_config = ModelConfig(model_name=MODEL_NAME)
model = GemmaModel(model_config)
sae = JumpReLUSAE.from_pretrained(sae_config, device=model_config.device)

### Configure Steering

In [ ]:
STEER_FEATURE = 0    # <-- fill in from heatmap / feature_map inspection
STEER_COEFF = 10.0   # <-- fill in (positive = amplify, negative = suppress)

prompt_text = sequences[sample_idx][:prompt_char_lens[sample_idx]]
inputs = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=True)["input_ids"].to(model_config.device)

print(f"Sample index:   {sample_idx}")
print(f"Steer feature:  {STEER_FEATURE}")
print(f"Steer coeff:    {STEER_COEFF}")
print(f"Prompt tokens:  {inputs.shape[1]}")
print(f"Prompt text:    {prompt_text[:200]}...")

### Baseline vs Steered Generation

In [ ]:
from src.utils.steering import generate_with_steering_and_capture

baseline_text, baseline_ids, baseline_sae_acts = generate_with_steering_and_capture(
    model, sae, inputs, sae_config.layer, STEER_FEATURE, coeff=None,
    max_new_tokens=256,
)

steered_text, steered_ids, steered_sae_acts = generate_with_steering_and_capture(
    model, sae, inputs, sae_config.layer, STEER_FEATURE, coeff=STEER_COEFF,
    max_new_tokens=256,
)

print(f"{'BASELINE':=^80}")
print(baseline_text)
print()
print(f"{'STEERED (feature={STEER_FEATURE}, coeff={STEER_COEFF})':=^80}")
print(steered_text)
print()
print(f"{'COMPARISON':=^80}")
print(f"Baseline length:  {len(baseline_ids)} tokens")
print(f"Steered length:   {len(steered_ids)} tokens")
print(f"Texts identical:  {baseline_text == steered_text}")